# Swarm reservoir tutorial

This notebook builds a complete swarm reservoir in four stages:

1. choose and inspect the **swarm dynamics**;
2. choose an **input signal and coupling** and see how behaviour changes;
3. choose the **observation layer**;
4. train the linear **readout** and inspect which observations mattered.

Companion notebooks:

- `Tutorial_ABM.ipynb`: agent-based swarm simulation and analysis;
- `Tutorial_ESN.ipynb`: conventional echo-state networks and diagnostics;
- `Tutorial_TDA.ipynb`: topological observations of swarm and reservoir states;
- `advanced/Existing_Model_Validation.ipynb`: validation of Jaeger, Couzin, Lymburn and Topaz models.


In [1]:
for d in ("FIGURES/ESN", "FIGURES/ABM", "FIGURES/swarmRC", "ANIMATIONS/ABM", "ANIMATIONS/swarmRC")
    mkpath(d)
end

## ▶ Design your swarm reservoir

The ▶ symbol here indicates two shortcuts in this notebook used for a talk that presented the overall research program. This first shortcut is the control panel; the second, **Complete pipeline at a glance**, displays the resulting pipeline. Students can ignore the symbol and run the notebook from top to bottom.

We include the presets required to run three published models:

| existing models | input coupling | default observation |
|---|---|---|
| `:lymburn_tutorial` interacting boids-style agents | Lorenz $(x,y)$ as a moving predator | Gaussian density and velocity responses |
| `:mizzi_tutorial` distinguishable territorial agents | delay-embedded Lorenz as moving prey | home-relative positions and velocities |
| `:lund_tutorial` two-state boids | scalar Lorenz as global temperature/speed | nine state-aware aggregate measurements |
| `:lund_paper` companion-code dynamics | scalar Lorenz as global temperature/speed | nine state-aware aggregate measurements |

The presets choose the swarm, its input mechanism and its default observation together. **Lymburn:** interchangeable agents are observed through group-level density and velocity fields. **Mizzi:** a unique home makes each agent's home-relative state directly readable. **Lund:** a slow dispersed/clustered state accompanies global scalar coupling to target speed.

These are paper-based teaching pipelines, not reproductions of every published experiment. **Mizzi:** MDL optimisation and the paper's evaluation protocol are omitted. **Lymburn:** the tutorial uses a rotationally symmetric force cap instead of the original MATLAB code's componentwise cap. **Lund:** `:lund_tutorial` is a recalibrated teaching version; `:lund_paper` matches the companion-code dynamics but retains this tutorial's Lorenz task and nine-feature observation. See [`MODEL_IMPLEMENTATION_COMPARISON.md`](./MODEL_IMPLEMENTATION_COMPARISON.md) for the detailed comparison.

Use `:custom` to change individual components. `run_mode=:results_only` skips optional outputs; use `:full` to generate them.

In [ ]:
# Choose a fixed tutorial pipeline. Then restart Julia and Run All.
student_choice = (
    pipeline = :lymburn_tutorial, # :lymburn_tutorial, :mizzi_tutorial,
                                  # :lund_tutorial, :lund_paper or :custom
    run_mode = :results_only,     # :results_only skips auxiliary outputs; :full shows everything
    custom = (
        swarm = :lymburn,                 # :lymburn, :mizzi or :lund
        input_signal = :lorenz,           # ready-to-run signal: :lorenz
        coupling = :predator_position,    # :predator_position, :prey_position,
                                          # :temperature_speed or :none (control)
        observation = :spatial_gaussian,  # :spatial_gaussian, :raw_positions,
                                          # :raw_state or :lund_aggregate
        lund_preset = :tutorial,           # :tutorial or :paper
    ),
)

pipeline_presets = (
    lymburn_tutorial = (swarm=:lymburn, input_signal=:lorenz, lund_preset=:tutorial,
        coupling=:predator_position, observation=:spatial_gaussian),
    mizzi_tutorial = (swarm=:mizzi, input_signal=:lorenz, lund_preset=:tutorial,
        coupling=:prey_position, observation=:raw_state),
    lund_tutorial = (swarm=:lund, input_signal=:lorenz, lund_preset=:tutorial,
        coupling=:temperature_speed, observation=:lund_aggregate),
    lund_paper = (swarm=:lund, input_signal=:lorenz, lund_preset=:paper,
        coupling=:temperature_speed, observation=:lund_aggregate),
)
student_choice.pipeline == :custom || student_choice.pipeline in propertynames(pipeline_presets) ||
    error("Unknown pipeline $(student_choice.pipeline). Choose :lymburn_tutorial, :mizzi_tutorial, :lund_tutorial, :lund_paper or :custom.")
choice = student_choice.pipeline == :custom ? student_choice.custom :
    getproperty(pipeline_presets, student_choice.pipeline)
student_choice.run_mode in (:results_only, :full) ||
    error("Choose run_mode=:results_only or :full.")
full_mode = student_choice.run_mode == :full
heading_colormap = :vikO  # cyclic, colour-vision-friendly heading scale

# Each swarm has a physical input interface and state representation.
implemented = (
    lymburn = (couplings=(:predator_position, :none),
               observations=(:spatial_gaussian, :raw_positions)),
    mizzi = (couplings=(:prey_position, :none),
             observations=(:raw_state,)),
    lund = (couplings=(:temperature_speed, :none),
            observations=(:lund_aggregate,)),
)

choice.swarm in propertynames(implemented) ||
    error("Unknown swarm $(choice.swarm). Choose :lymburn, :mizzi or :lund.")
choice.input_signal == :lorenz ||
    error("Only :lorenz currently has a defined representation, scale and sampling adapter in these ready-to-run pipelines. Other generators are available in TIME_SERIES/my_systems.jl for extension.")
valid = getproperty(implemented, choice.swarm)
choice.coupling in valid.couplings ||
    error("For swarm=$(choice.swarm), choose coupling from $(valid.couplings).")
choice.observation in valid.observations ||
    error("For swarm=$(choice.swarm), choose observation from $(valid.observations).")
choice.coupling != :none ||
    @warn "coupling=:none is an undriven control: the input is generated but cannot enter the swarm."

selected_observation(swarm, baseline) =
    choice.swarm == swarm ? choice.observation : baseline

# Model parameters used by the selected complete pipeline.
swarm_pipeline = (
    dynamics = (
        model = choice.swarm,
        lymburn = (preset=:critical, N=200, dt=0.02, view_halfwidth=10.0),
        mizzi = (N=30, lag=7, home_source=:training_embedding, dt=0.02),
        lund = choice.lund_preset == :paper ?
            (preset=:paper, N=200, L=512.0, dt=0.1, threshold=3.0, hysteresis=0.05) :
            (preset=:tutorial, N=200, L=256.0, dt=0.1, threshold=1.10, hysteresis=0.10),
    ),
    input = (
        system = choice.input_signal,
        lymburn = (coupling=choice.swarm == :lymburn ? choice.coupling : :predator_position, representation=:xy_projection,
                   transform=:zscore, target_std=2.0),
        mizzi = (coupling=choice.swarm == :mizzi ? choice.coupling : :prey_position, representation=:scalar_delay_embedding, scale_divisor=7.34),
        lund = (coupling=choice.swarm == :lund ? choice.coupling : :temperature_speed, representation=:scalar,
                transform=:zscore, target_std=0.5, temperature_gain=0.30),
    ),
    observation = (
        lymburn = (compare=(:raw_positions, :spatial_gaussian),
                   selected=selected_observation(:lymburn, :spatial_gaussian), M=200, kneigh=5),
        mizzi = (selected=selected_observation(:mizzi, :raw_state),),
        lund = (selected=selected_observation(:lund, :lund_aggregate),),
    ),
    task = (
        lymburn = (target=:future_input_x, horizon_time=0.5),
        mizzi = (target=:next_scalar_sample, horizon_steps=1),
        lund = (target=:future_scalar, horizon_steps=1),
    ),
    animation = (every=1, lund_steps=750, overwrite=false),
)

swarm_pipeline

## 1. Inspect the swarm dynamics

This section loads the chosen swarm, checks its operating regime and visualises its undriven behaviour before an input is applied.



In [ ]:
selected_model = swarm_pipeline.dynamics.model
selected_model in (:lymburn, :mizzi, :lund) ||
    error("The resolved swarm must be :lymburn, :mizzi or :lund; got $selected_model.")

if selected_model == :lymburn
    include("SWARM_RC/load_SwarmRC_Lymburn.jl")
elseif selected_model == :mizzi
    include("SWARM_RC/load_SwarmRC_Mizzi.jl")
else
    include("SWARM_RC/load_SwarmRC_Lund.jl")
end
include("SWARM_RC/my_swarmRC_plotting.jl")
include("SWARM_RC/my_predator.jl")
include("TIME_SERIES/my_systems.jl")
include("TIME_SERIES/my_time_series_analysis.jl");

### Use model-specific evidence to choose an operating regime

For **Lymburn**, the saved parameter sweep varies repulsion $K_r$ and alignment $K_a$. It supports the selected `:critical` preset: point B, $K_r=2$, $K_a=0.01$, a repeatable region with nontrivial collective motion. 

![Lymburn parameter sweep: behaviour and computation](FIGURES/swarmRC/lymburn_sweep_heatmap.png)

The complete **Lymburn** sweep remains in `advanced/Existing_Model_Validation.ipynb`.

For **Mizzi**, agents do not interact, so there is no collective-motion regime map. The main structural choice is the home arrangement. Mizzi et al. compared two random baselines: homes sampled from points of the embedded training trajectory, and homes sampled uniformly from the square containing that trajectory. Sampling from the trajectory produced higher consistency capacity because more territories received input. Uniform sampling sometimes produced longer forecasts, showing that consistency alone does not determine task performance. The paper later optimised homes with an evolutionary MDL procedure. This tutorial implements both random baselines (`mizzi_homes_from_input` and `mizzi_homes_uniform`) and selects the embedded-training-point method; it excludes MDL optimisation.

For **Lund**, this smaller system can collapse to an all-dispersed state when
the switching threshold is too high. The tutorial uses $\rho=1.10$ and $h=0.10$: with the seeded input this keeps both states active after washout. Blue agents are dispersed and red agents are clustered. The value $\rho=1.10$ is a tutorial setting, not a paper calibration. The paper's memory experiments use $N\ge800$ in a $512\times512$ torus; this tutorial uses $N=200$, $L=256$ to retain agent density.


In [ ]:
if full_mode
# Inspect an undriven instance of the selected dynamics.
dt = getproperty(getproperty(swarm_pipeline.dynamics, selected_model), :dt)
if selected_model == :lymburn
    cfg = swarm_pipeline.dynamics.lymburn
    P_demo = Lymburn_params_from_preset(cfg.preset; N=cfg.N)
    simcfg_demo = SimulationConfig(steps=1000, dt=dt, seed=1)
    out_undriven = simulate_Lymburn_2d(simcfg_demo, P_demo; show_progress=false)
    selected_regime = (model=:lymburn, preset=cfg.preset,
        Kr=P_demo.Kr, Ka=P_demo.Ka, N=P_demo.N, dt=dt)
elseif selected_model == :mizzi
    cfg = swarm_pipeline.dynamics.mizzi
    demo_homes = mizzi_homes_uniform(cfg.N; halfwidth=2.5, rng=MersenneTwister(1))
    P_demo = MizziParams(demo_homes)
    # With no prey, an exact home/zero-velocity start is stationary. Perturb it
    # slightly so this short demo shows homing and damping back to equilibrium.
    simcfg_demo = SimulationConfig(steps=500, dt=dt, seed=1)
    out_undriven = simulate_Mizzi_2d(simcfg_demo, P_demo;
        position_noise=0.25, velocity_noise=0.10, show_progress=false)
    selected_regime = (model=:mizzi, N=P_demo.N, dt=dt,
        kh=P_demo.kh, kd=P_demo.kd, kf=P_demo.kf,
        homes=:uniform_demo_only)
else
    cfg = swarm_pipeline.dynamics.lund
    P_demo = Lund_params_from_preset(cfg.preset; N=cfg.N)
    out_undriven = simulate_Lund_2d(P_demo;
        steps=swarm_pipeline.animation.lund_steps, rng=MersenneTwister(1))
    selected_regime = (model=:lund, N=P_demo.N, L=P_demo.L, dt=P_demo.dt,
        threshold=P_demo.energy_threshold, hysteresis=P_demo.hysteresis_factor,
        note=cfg.preset == :paper ? "companion-code default dynamics" :
            "recalibrated tutorial dynamics")
end

selected_regime
else
    println("Skipped in results-only mode; set student_choice.run_mode=:full to generate this optional result.")
end


In [ ]:
if full_mode
snapshot_times = selected_model == :mizzi ? [0.0, 0.5, 2.0, 10.0] :
    (selected_model == :lund ? [0.0, 5.0, 15.0, 30.0] : [0.0, 5.0, 7.0, 10.0])
frames = [argmin(abs.(out_undriven.t .- t)) for t in snapshot_times]

# The cyclic heading map is defined in the switchboard so fast mode can also use it.
fig_swarm = Figure(size=(1050, 330))
for (j, k) in enumerate(frames)
    ax = Axis(fig_swarm[1, j],
        title="t=$(round(out_undriven.t[k], digits=1)), ΦR=$(round(out_undriven.rotation[k], digits=2))",
        aspect=DataAspect())
    selected_model == :mizzi && plot_Mizzi_territories!(ax, P_demo)
    if selected_model == :lund
        pts = out_undriven.pos_hist[k]
        colors = [s == 0 ? :dodgerblue : :firebrick for s in out_undriven.state_hist[k]]
        scatter!(ax, [p.x for p in pts], [p.y for p in pts]; color=colors, markersize=6)
        xlims!(ax, 0, P_demo.L); ylims!(ax, 0, P_demo.L)
    else
        plot_swarm_frame!(ax, out_undriven.pos_hist, out_undriven.vel_hist, k;
            trail_len=25, agent_ms=6.0,
            color_by_heading=selected_model == :lymburn, colormap=heading_colormap,
            agent_color=:firebrick)
        if selected_model == :mizzi && j == 1
            scatter!(ax, [NaN], [NaN]; color=:firebrick, markersize=6, label="agents")
            axislegend(ax; position=:lt)
        end
        if selected_model == :lymburn
            pos = out_undriven.pos_hist[k]
            vel = out_undriven.vel_hist[k]
            stride = 2
            arrow_indices = collect(1:stride:length(pos))
            arrow_positions = [Point2f(pos[i].x, pos[i].y) for i in arrow_indices]
            arrow_directions = [begin
                speed = hypot(vel[i].x, vel[i].y)
                Vec2f(0.8 * vel[i].x / max(speed, eps()),
                      0.8 * vel[i].y / max(speed, eps()))
            end for i in arrow_indices]
            arrow_headings = [atan(vel[i].y, vel[i].x) for i in arrow_indices]
            arrows2d!(ax, arrow_positions, arrow_directions;
                color=arrow_headings, colormap=heading_colormap, colorrange=(-pi, pi),
                tiplength=5, shaftwidth=1.0)
        end
        h = selected_model == :lymburn ? swarm_pipeline.dynamics.lymburn.view_halfwidth : P_demo.plot_extent
        xlims!(ax, -h, h); ylims!(ax, -h, h)
    end
end
if selected_model == :lymburn
    Colorbar(fig_swarm[2, 1:4]; limits=(-pi, pi), colormap=heading_colormap,
        vertical=false, ticks=([-pi, -pi/2, 0, pi/2, pi],
        ["−π", "−π/2", "0", "π/2", "π"]), label="heading angle")
end
save("FIGURES/swarmRC/selected_$(selected_model)_undriven_snapshots.png", fig_swarm)
fig_swarm
else
    println("Skipped in results-only mode; set student_choice.run_mode=:full to generate this optional result.")
end


### Optional animations

Animations are generated only with `run_mode=:full` and reused when cached. In the `swarm_pipeline.animation` settings, set `overwrite=true` to regenerate them or change `every` to alter frame sampling. Keep the model timestep fixed.

In [ ]:
if full_mode
# Use an animation to check how the selected dynamics evolve through time.
# Convert the saved agent vectors into the 2×N matrices expected by the animator.
xy_matrix(points) = [getproperty(p, coordinate) for coordinate in (:x, :y), p in points]
states_undriven = [
    (P=xy_matrix(out_undriven.pos_hist[k]), V=xy_matrix(out_undriven.vel_hist[k]))
    for k in eachindex(out_undriven.pos_hist)
]
direction_tag = selected_model == :lymburn ? "short_heading_arrows_view10" :
    (selected_model == :lund ? "short_state_arrows" : "short_direction_arrows")
model_cache_tag = selected_model == :lund ? "lund_$(swarm_pipeline.dynamics.lund.preset)" : string(selected_model)
undriven_animation_path = "ANIMATIONS/swarmRC/selected_$(model_cache_tag)_undriven_$(direction_tag)_$(length(states_undriven))steps_every$(swarm_pipeline.animation.every).mp4"
if swarm_pipeline.animation.overwrite || !isfile(undriven_animation_path)
    animation_halfwidth = selected_model == :lymburn ?
        swarm_pipeline.dynamics.lymburn.view_halfwidth :
        (selected_model == :lund ? P_demo.L / 2 : P_demo.plot_extent)
    animation_L = selected_model == :lund ? P_demo.L : 2animation_halfwidth
    animation_center = selected_model == :lund ? (0.0, 0.0) : (-animation_halfwidth, -animation_halfwidth)
    animate_from_states(states_undriven, animation_L;
        every=swarm_pipeline.animation.every,
        center=animation_center,
        agent_state_hist=selected_model == :lund ? out_undriven.state_hist : nothing,
        show_agent_dirs=true, dir_stride=selected_model == :mizzi ? 1 : 2,
        dir_len=0.018animation_L,
        color_by_heading=selected_model == :lymburn, heading_colormap=heading_colormap,
        show_agent_trails=true, agent_trail_len=8, agent_trail_color=(:grey40, 0.12),
        agent_ms=7.0, agent_color=selected_model == :mizzi ? :firebrick : :darkcyan, show_grid=false,
        territory_homes=selected_model == :mizzi ? P_demo.homes : nothing,
        savepath=undriven_animation_path)
else
    println("Using saved animation: $undriven_animation_path")
end
display("text/html", """
<video controls loop muted style=\"width: min(900px, 100%); height: auto;\">
  <source src=\"$(undriven_animation_path)\" type=\"video/mp4\">
  Your browser cannot display this video. Open $(undriven_animation_path) directly.
</video>
""")
undriven_animation_path
else
    println("Skipped in results-only mode; set student_choice.run_mode=:full to generate this optional result.")
end


## 2. Choose the input signal and coupling

Each preset uses the input coupling defined for that model:

- **Lymburn:** a rescaled Lorenz $(x,y)$ projection is a local spatial driver through `PredatorCoupling`. The two projected coordinates are the moving predator.
- **Lund:** a scalar Lorenz $x$ record is a system-wide temperature that maps to the target speed $s_0(1+g u(n))$ through `TemperatureSpeedCoupling`; the two-state boids turn that common input into heterogeneous, hysteretic responses.
- **Mizzi:** the scalar Lorenz record becomes prey position $p(t)=(u(t-\tau),u(t))$, and `MizziPreyCoupling` drives only the agent whose Voronoi cell contains the prey and agents in adjacent cells.

The Lorenz ODE is integrated with a maximum internal step of $0.01$ and
interpolated at the selected swarm update interval. **Lymburn** and **Mizzi** use $\Delta t=0.02$; **Lund** uses $\Delta t=0.1$. They share the same Lorenz equations, parameters, random seed and initial-condition rule, but not the same sampled sequence. **Lymburn** predicts $0.5$ time units ahead; **Mizzi** and **Lund** predict one model step ahead.

Rössler, hyper-Rössler, logistic and Mackey–Glass generators are available in `TIME_SERIES/my_systems.jl`. To use one here, define its representation, scale, sampling interval and prediction target.


In [ ]:
# Build the selected input, target and reservoir.
observation_choice = getproperty(getproperty(swarm_pipeline.observation, selected_model), :selected)
shift, train_len, predict_len, washout = selected_model == :lund ?
    (100, 800, 300, 100) : (200, 3000, 800, 200)
dt = getproperty(getproperty(swarm_pipeline.dynamics, selected_model), :dt)
rng_lorenz = MersenneTwister(7)
input_duration = max(120.0, dt * (shift + train_len + predict_len + 100))
lor = lorenz_data(rng=rng_lorenz, tspan=(0.0, input_duration), dtmax=0.01)
tgrid = collect(0.0:dt:input_duration)
X3 = Array(lor.sol(tgrid))

if selected_model == :lymburn
    input_cfg = swarm_pipeline.input.lymburn
    U_full = zscore_rescale(X3[1:2, :]; target_std=input_cfg.target_std)
    horizon = round(Int, swarm_pipeline.task.lymburn.horizon_time / dt)
    Ttot = size(U_full, 2) - horizon
    U = U_full[:, 1:Ttot]
    target = U_full[1, (1:Ttot) .+ horizon]

    dyn_cfg = swarm_pipeline.dynamics.lymburn
    P_selected = Lymburn_params_from_preset(dyn_cfg.preset; N=dyn_cfg.N)
    selected_coupling = input_cfg.coupling == :predator_position ?
        PredatorCoupling(rp=P_selected.rp) : NoCoupling()
    res_selected = build_Lymburn_reservoir(P_selected, dt;
        coupling=selected_coupling, rng=MersenneTwister(1))
elseif selected_model == :mizzi
    input_cfg = swarm_pipeline.input.mizzi
    u_scalar = vec(X3[1, :]) ./ input_cfg.scale_divisor
    lag = swarm_pipeline.dynamics.mizzi.lag
    U_full = vcat(permutedims(u_scalar[1:end-lag]),
                  permutedims(u_scalar[1+lag:end]))
    horizon = swarm_pipeline.task.mizzi.horizon_steps
    Ttot = size(U_full, 2) - horizon
    U = U_full[:, 1:Ttot]
    target = U_full[2, (1:Ttot) .+ horizon]

    dyn_cfg = swarm_pipeline.dynamics.mizzi
    home_stop = min(size(U, 2), shift + train_len)
    selected_coupling = input_cfg.coupling == :prey_position ?
        MizziPreyCoupling() : NoCoupling()
    homes = mizzi_homes_from_input(U[:, 1:home_stop], dyn_cfg.N; rng=MersenneTwister(1))
    P_selected = MizziParams(homes)
    res_selected = build_Mizzi_reservoir(P_selected, dt;
        coupling=selected_coupling, rng=MersenneTwister(1))
else
    input_cfg = swarm_pipeline.input.lund
    U_full = zscore_rescale(X3[1:1, :]; target_std=input_cfg.target_std)
    horizon = swarm_pipeline.task.lund.horizon_steps
    Ttot = size(U_full, 2) - horizon
    U = U_full[:, 1:Ttot]
    target = U_full[1, (1:Ttot) .+ horizon]
    dyn_cfg = swarm_pipeline.dynamics.lund
    P_selected = Lund_params_from_preset(dyn_cfg.preset; N=dyn_cfg.N)
    selected_coupling = input_cfg.coupling == :temperature_speed ?
        TemperatureSpeedCoupling(base_speed=P_selected.base_speed,
            gain=P_selected.temperature_gain, min_speed=P_selected.min_speed,
            max_speed=P_selected.max_speed) : NoCoupling()
    res_selected = build_Lund_reservoir(P_selected;
        coupling=selected_coupling, rng=MersenneTwister(1))
end

input_steps = [norm(U[:, k+1] - U[:, k]) for k in 1:size(U, 2)-1]
sampling_summary = (
    lorenz_seed=7, integration_dtmax=0.01, sample_interval=dt,
    target_horizon_steps=horizon, target_horizon_time=horizon * dt,
    median_input_step=median(input_steps), maximum_input_step=maximum(input_steps),
    animation_frame_interval=dt * swarm_pipeline.animation.every,
    note="The seeded Lorenz system is shared, but sampling and prediction horizons are model-specific.",
)

(pipeline=student_choice.pipeline, run_mode=student_choice.run_mode, swarm=selected_model, input_signal=choice.input_signal,
 coupling=choice.coupling, observation=observation_choice,
 input_size=size(U), target_length=length(target), agents=P_selected.N,
 sampling=sampling_summary)

In [ ]:
if full_mode && !(res_selected.coupling isa NoCoupling)
# Compare the same selected reservoir with coupling disabled/enabled.
order_fn = selected_model == :lymburn ? Lymburn_order_parameters_2d :
    (selected_model == :mizzi ? Mizzi_order_parameters_2d : order_parameters_2d)
disp_fn = selected_model == :lymburn ? displacement_nonperiodic_2d :
    (selected_model == :mizzi ? mizzi_displacement_2d : lund_displacement_from_to)
diagnostic_extent = selected_model == :lund ? P_selected.L : P_selected.plot_extent

spatial_driver = selected_model in (:lymburn, :mizzi)
drive_fn = spatial_driver ? run_offline_predator_drive! : run_offline_input_drive!
demo_steps = selected_model == :lund ? swarm_pipeline.animation.lund_steps : 1000
res_undriven = clone_reservoir(res_selected)
res_undriven.coupling = NoCoupling()
out_undriven_selected = drive_fn(res_undriven, U[:, 1:demo_steps];
    dt=dt, rng=MersenneTwister(13), order_parameters=order_fn,
    displacement=disp_fn, L=diagnostic_extent, show_progress=false)

reset!(res_selected; rng=MersenneTwister(1))
out_driven_demo = drive_fn(res_selected, U[:, 1:demo_steps];
    dt=dt, rng=MersenneTwister(13), order_parameters=order_fn,
    displacement=disp_fn, L=diagnostic_extent, show_progress=false)

fig_change = Figure(size=(1050, 360))
lymburn_view = swarm_pipeline.dynamics.lymburn.view_halfwidth
axes_change = Axis[]
for (slot, st, title) in ((1, out_undriven_selected.states[end], "Coupling disabled"),
                          (2, out_driven_demo.states[end], "Selected input coupling"))
    ax = Axis(fig_change[1, slot], title=title, aspect=DataAspect())
    push!(axes_change, ax)
    selected_model == :mizzi && plot_Mizzi_territories!(ax, P_selected)
    if selected_model == :lymburn
        heading = atan.(st.V[2, :], st.V[1, :])
        scatter!(ax, st.P[1, :], st.P[2, :]; color=heading,
            colormap=heading_colormap, colorrange=(-pi, pi), markersize=6)
        stride = 2
        arrow_indices = collect(1:stride:P_selected.N)
        arrow_positions = [Point2f(st.P[1, i], st.P[2, i]) for i in arrow_indices]
        arrow_directions = [begin
            speed = hypot(st.V[1, i], st.V[2, i])
            Vec2f(0.8 * st.V[1, i] / max(speed, eps()),
                  0.8 * st.V[2, i] / max(speed, eps()))
        end for i in arrow_indices]
        arrow_headings = heading[arrow_indices]
        arrows2d!(ax, arrow_positions, arrow_directions;
            color=arrow_headings, colormap=heading_colormap, colorrange=(-pi, pi),
            tiplength=5, shaftwidth=1.0)
        xlims!(ax, -lymburn_view, lymburn_view); ylims!(ax, -lymburn_view, lymburn_view)
    elseif selected_model == :mizzi
        scatter!(ax, st.P[1, :], st.P[2, :]; color=:firebrick, markersize=6, label="agents")
    else
        colors = [s == 0 ? :dodgerblue : :firebrick for s in st.agent_state]
        scatter!(ax, st.P[1, :], st.P[2, :]; color=colors, markersize=6)
        xlims!(ax, 0, P_selected.L); ylims!(ax, 0, P_selected.L)
    end
end
if spatial_driver
    pred = out_driven_demo.pred_hist[end]
    scatter!(axes_change[2], [pred.x], [pred.y];
        color=selected_model == :mizzi ? :dodgerblue : :red,
        marker=:star5, markersize=14,
        label=selected_model == :mizzi ? "prey" : "predator")
end
if selected_model == :mizzi
    axislegend.(axes_change; position=:lt)
elseif selected_model == :lymburn
    Colorbar(fig_change[2, 1:2]; limits=(-pi, pi), colormap=heading_colormap,
        vertical=false, ticks=([-pi, -pi/2, 0, pi/2, pi],
        ["−π", "−π/2", "0", "π/2", "π"]), label="heading angle")
end
ax3 = Axis(fig_change[1, 3], xlabel="time", ylabel="order parameter",
    title="Input-altered dynamics")
lines!(ax3, out_driven_demo.t, out_driven_demo.rotation; label="rotation")
lines!(ax3, out_driven_demo.t, out_driven_demo.polarisation; label="polarisation")
if selected_model == :lund && res_selected.coupling isa TemperatureSpeedCoupling
    speed_trace = target_speed.(Ref(res_selected.coupling), vec(out_driven_demo.input))
    lines!(ax3, out_driven_demo.t, speed_trace ./ maximum(speed_trace);
        label="target speed (scaled)", linestyle=:dash)
end
axislegend(ax3)
save("FIGURES/swarmRC/selected_$(selected_model)_coupling_comparison.png", fig_change)
fig_change
else
    println(full_mode ? "NoCoupling selected; there is no driven comparison to plot." :
        "Skipped in results-only mode; set student_choice.run_mode=:full to generate this optional result.")
end


In [ ]:
if full_mode && !(res_selected.coupling isa NoCoupling)
influence_radius = selected_model == :lymburn ? P_selected.rp : 0.0
coupling_tag = selected_model == :lymburn ? :predator_position :
    (selected_model == :mizzi ? :prey_position : :temperature_speed)
direction_tag = selected_model == :lymburn ? "short_heading_arrows_view10" :
    (selected_model == :lund ? "short_state_arrows" : "short_direction_arrows")
model_cache_tag = selected_model == :lund ? "lund_$(swarm_pipeline.dynamics.lund.preset)" : string(selected_model)
driven_animation_path = "ANIMATIONS/swarmRC/selected_$(model_cache_tag)_$(coupling_tag)_driven_$(direction_tag)_$(demo_steps)steps_every$(swarm_pipeline.animation.every).mp4"
if swarm_pipeline.animation.overwrite || !isfile(driven_animation_path)
    animation_halfwidth = selected_model == :lymburn ?
        swarm_pipeline.dynamics.lymburn.view_halfwidth :
        (selected_model == :lund ? P_selected.L / 2 : P_selected.plot_extent)
    animation_L = selected_model == :lund ? P_selected.L : 2animation_halfwidth
    animation_center = selected_model == :lund ? (0.0, 0.0) : (-animation_halfwidth, -animation_halfwidth)
    animate_from_states(out_driven_demo.states, animation_L;
        every=swarm_pipeline.animation.every,
        center=animation_center,
        agent_state_hist=selected_model == :lund ? [st.agent_state for st in out_driven_demo.states] : nothing,
        pred_hist=spatial_driver ? out_driven_demo.pred_hist : nothing,
        predator_color=selected_model == :mizzi ? :dodgerblue : :red,
        show_influence_radius=selected_model == :lymburn && spatial_driver,
        show_agent_dirs=true, dir_stride=selected_model == :mizzi ? 1 : 2,
        dir_len=0.018animation_L,
        color_by_heading=selected_model == :lymburn, heading_colormap=heading_colormap,
        show_agent_trails=true, agent_trail_len=8, agent_trail_color=(:grey40, 0.12),
        agent_ms=7.0, agent_color=selected_model == :mizzi ? :firebrick : :darkcyan, show_grid=false,
        territory_homes=selected_model == :mizzi ? P_selected.homes : nothing,
        savepath=driven_animation_path, rp=influence_radius)
else
    println("Using saved animation: $driven_animation_path")
end
display("text/html", """
<video controls loop muted style=\"width: min(900px, 100%); height: auto;\">
  <source src=\"$(driven_animation_path)\" type=\"video/mp4\">
  Your browser cannot display this video. Open $(driven_animation_path) directly.
</video>
""")
driven_animation_path
else
    println(full_mode ? "NoCoupling selected; no driven animation is generated." :
        "Skipped in results-only mode; set student_choice.run_mode=:full to generate this optional result.")
end


## 3. Choose what is observable

Each preset has a paper-aligned observation:

- **Lymburn:** Gaussian density and velocity fields, compared with the raw `2N` positions;
- **Mizzi:** the `4N` home-relative positions and velocities, because fixed homes distinguish the agents;
- **Lund:** nine state-aware aggregate statistics, including clustered fraction and state-specific speeds.


In [ ]:
if observation_choice == :spatial_gaussian
    obs_cfg = swarm_pipeline.observation.lymburn
    observation_selected = build_observation_layer!(
        res_selected, U[:, 1:shift+train_len];
        washout=washout, rng=MersenneTwister(2), method=:spatial_gaussian,
        M=obs_cfg.M, kneigh=obs_cfg.kneigh, show_progress=false)
elseif observation_choice in (:raw_positions, :raw_state, :lund_aggregate)
    observation_selected = nothing
else
    error("Unsupported observation choice: $observation_choice")
end

(model=selected_model, observation=observation_choice,
 feature_count=length(feature_map(res_selected, observation_selected)))

## 4. Validate the pipeline

Run the structural check before training. Errors identify incompatible choices; warnings identify cases that need inspection before training.

`validate_pipeline` checks:

- finite input and target data with the required dimensions;
- a coupling and observation supported by the selected swarm;
- sensible observation-layer parameters;
- input scale, sampling interval and actuator range;
- non-constant, sufficiently varied observations from a short driven run;
- enough predicted coordinates for any claimed autonomous task.


In [ ]:
pipeline_check = validate_pipeline(res_selected, U;
    observation=observation_choice, obs=observation_selected,
    target=target, prediction_mode=:teacher_forced,
    dt_input=dt, probe_steps=300,
    rng=MersenneTwister(90))
print_pipeline_validation(pipeline_check)
pipeline_check.valid || error("Invalid swarm-reservoir pipeline; see report above.")

## 5. Train the readout

Only the linear ridge readout is trained; the swarm and observation layer remain fixed. Performance is measured by prediction of unseen data while the true input continues to drive the swarm. **Lymburn:** compare raw positions with kernels. **Mizzi:** use the direct `4N` state. **Lund:** use nine aggregate features.

In [ ]:
function collect_selected(res, obs)
    X, _ = collect_features(res, U[:, 1:shift+train_len+predict_len];
        rng=MersenneTwister(3), log_raw=false, reset_res=true,
        feature_fn=feature_map, obs=obs, show_progress=false)
    return X
end

# Collect both Lymburn observations from one physical trajectory. This avoids
# integrating the same deterministic 200-agent swarm twice.
function collect_selected_pair(res, obs)
    Useq = U[:, 1:shift+train_len+predict_len]
    rng = MersenneTwister(3)
    reset!(res; rng=rng)
    reservoir_step!(res, view(Useq, :, 1); rng=rng)
    xraw, xobs = feature_map(res, nothing), feature_map(res, obs)
    Xraw = Matrix{Float64}(undef, length(xraw), size(Useq, 2))
    Xobs = Matrix{Float64}(undef, length(xobs), size(Useq, 2))
    Xraw[:, 1], Xobs[:, 1] = xraw, xobs
    for t in 2:size(Useq, 2)
        reservoir_step!(res, view(Useq, :, t); rng=rng)
        Xraw[:, t] = feature_map(res, nothing)
        Xobs[:, t] = feature_map(res, obs)
    end
    return Xraw, Xobs
end

function fit_selected_R(Xfeat; ridge_grid)
    Y = reshape(target[1:shift+train_len+predict_len], 1, :)
    Xtr = Xfeat[:, shift+washout+1:shift+train_len]
    Ytr = Y[:, shift+washout+1:shift+train_len]
    Xte = Xfeat[:, shift+train_len+1:shift+train_len+predict_len]
    Yte = Y[:, shift+train_len+1:shift+train_len+predict_len]
    Wout, μx, σx, best_λ, _ = train_readout_cv(
        Xtr, Ytr; ridge_grid=ridge_grid, n_folds=5, gap=25)
    Yhat = apply_readout(Wout, Xte, μx, σx)
    return cor(vec(Yhat), vec(Yte)), Yhat, Yte, best_λ, Wout
end

if selected_model == :lymburn && observation_choice == :spatial_gaussian
    Xraw, Xselected = collect_selected_pair(res_selected, observation_selected)
    R_raw, Yhat_raw, Yte_selected, λ_raw, Wout_raw = fit_selected_R(
        Xraw; ridge_grid=10.0 .^ (-2:1:6))
    R_selected, Yhat_selected, _, λ_selected, Wout_selected = fit_selected_R(
        Xselected; ridge_grid=10.0 .^ (-3:1:5))
    println("Lymburn raw positions: R = ", round(R_raw, digits=3), " (λ=", λ_raw, ")")
    println("Lymburn kernels:       R = ", round(R_selected, digits=3), " (λ=", λ_selected, ")")
elseif selected_model == :lymburn && observation_choice == :raw_positions
    Xselected = collect_selected(res_selected, nothing)
    R_selected, Yhat_selected, Yte_selected, λ_selected, Wout_selected = fit_selected_R(
        Xselected; ridge_grid=10.0 .^ (-2:1:6))
    R_raw, Yhat_raw = R_selected, Yhat_selected
    println("Lymburn raw positions: R = ", round(R_selected, digits=3), " (λ=", λ_selected, ")")
elseif selected_model == :mizzi
    Xselected = collect_selected(res_selected, nothing)
    R_selected, Yhat_selected, Yte_selected, λ_selected, Wout_selected = fit_selected_R(
        Xselected; ridge_grid=10.0 .^ (-8:1:2))
    R_raw, Yhat_raw = R_selected, Yhat_selected
    println("Mizzi direct 4N state: R = ", round(R_selected, digits=3), " (λ=", λ_selected, ")")
elseif selected_model == :lund
    Xselected = collect_selected(res_selected, nothing)
    R_selected, Yhat_selected, Yte_selected, λ_selected, Wout_selected = fit_selected_R(
        Xselected; ridge_grid=10.0 .^ (-8:1:2))
    R_raw, Yhat_raw = R_selected, Yhat_selected
    println("Lund state-aware aggregates: R = ", round(R_selected, digits=3),
        " (λ=", λ_selected, ")")
else
    error("Unknown selected model: $selected_model")
end

# Preserve a time-aligned state for the pipeline summary before later
# diagnostics reset or advance res_selected. For Lund, choose the test-time
# state whose clustered fraction is closest to 0.5 so both internal states
# are visible when they occur anywhere in the analysed post-washout trajectory.
pipeline_snapshot_index = min(shift + train_len + predict_len, size(U, 2))
if selected_model == :lund
    snapshot_candidates = shift + washout + 1:pipeline_snapshot_index
    clustered_fraction = vec(Xselected[6, snapshot_candidates])
    pipeline_snapshot_index = snapshot_candidates[argmin(abs.(clustered_fraction .- 0.5))]
    pipeline_snapshot_res = clone_reservoir(res_selected)
    snapshot_rng = MersenneTwister(3)
    reset!(pipeline_snapshot_res; rng=snapshot_rng)
    for t in 1:pipeline_snapshot_index
        reservoir_step!(pipeline_snapshot_res, view(U, :, t); rng=snapshot_rng)
    end
else
    pipeline_snapshot_res = clone_reservoir(res_selected)
end
pipeline_snapshot_time = (pipeline_snapshot_index - 1) * dt

In [ ]:
if selected_model == :lymburn
    test_target_indices = (shift+train_len+1:shift+train_len+predict_len) .+ horizon
    lorenz_x_test = U_full[1, test_target_indices]
    @assert vec(Yte_selected) ≈ lorenz_x_test "Plotted target is not the expected future Lorenz x signal."
end
fig_pred = Figure(size=(900, 350))
ax = Axis(fig_pred[1, 1], xlabel="test sample", ylabel="target",
    title="$(uppercasefirst(String(selected_model))) reservoir: prediction on unseen test data")
lines!(ax, 1:length(Yte_selected), vec(Yte_selected);
    label="actual Lorenz x", color=:darkcyan, linewidth=2)
if selected_model == :lymburn && observation_choice == :spatial_gaussian
    lines!(ax, 1:length(Yhat_raw), vec(Yhat_raw); label="raw-position prediction", color=:grey55, linestyle=:dot)
    lines!(ax, 1:length(Yhat_selected), vec(Yhat_selected); label="Gaussian prediction", color=:black, linestyle=:dash)
elseif selected_model == :lymburn
    lines!(ax, 1:length(Yhat_selected), vec(Yhat_selected); label="prediction", color=:black, linestyle=:dash)
elseif selected_model == :mizzi
    lines!(ax, 1:length(Yhat_selected), vec(Yhat_selected); label="prediction", color=:black, linestyle=:dash)
elseif selected_model == :lund
    lines!(ax, 1:length(Yhat_selected), vec(Yhat_selected); label="prediction", color=:black, linestyle=:dash)
else
    error("Unknown selected model: $selected_model")
end
axislegend(ax; position=:rb)
save("FIGURES/swarmRC/selected_$(selected_model)_$(observation_choice)_prediction.png", fig_pred)
fig_pred

## 6. Check memory, separability and stability

The observations should separate input histories, retain task-relevant memory and remain stable enough for reproducible readout. These diagnostics should be interpreted together:

- **memory:** how well a linear readout recovers earlier values of the actual input. The memory curve is task-conditioned linear recall, not the classic input-independent memory capacity measured with an i.i.d. random driver;
- **separability:** whether three different input histories produce distinct observed states. Separability distances are standardised by each feature's training variation and by feature dimension;
- **stability:** whether two copies reconverge after a small perturbation to the physical swarm state. Stability uses identical initial conditions and identical random forcing in both replicas.


In [ ]:
# Reuse the trajectory already collected for training wherever possible.
diag_steps = min(shift + train_len, size(U, 2), size(Xselected, 2))
diag_U = Matrix(U[:, 1:diag_steps])
diag_X = Matrix(Xselected[:, 1:diag_steps])
memory_diag = memory_capacity_curve_from_features(diag_X, diag_U;
    input_dim=1, maxlag=50, washout=washout, ridgeλ=1e-4)

# Three equal-length Lorenz histories, all begun from the same swarm state.
sep_length = min(400, fld(size(U, 2), 4))
sep_starts = (1, sep_length + 1, 2sep_length + 1)
separability_inputs = [Matrix(U[:, s:s+sep_length-1]) for s in sep_starts]
scale_range = shift + washout + 1:shift + train_len
diagnostic_feature_scale = vec(std(Xselected[:, scale_range]; dims=2)) .+ 1e-12
separability_diag = sequence_separability(res_selected, separability_inputs;
    washout=min(washout, sep_length ÷ 3), rng=MersenneTwister(31),
    obs=observation_selected, feature_scale=diagnostic_feature_scale,
    normalise_dimension=true, compute_summary=false, show_progress=false)

# Perturb the continuous physical state by roughly one part per million.
stability_steps = min(600, size(U, 2))
physical_state = state_vector(res_selected, observation_selected)
stability_ε = 1e-6 * max(norm(physical_state), sqrt(length(physical_state)))
stability_diag = perturbation_stability(res_selected, Matrix(U[:, 1:stability_steps]);
    ε=stability_ε, warmup=min(washout, stability_steps ÷ 3),
    rng=MersenneTwister(32), obs=observation_selected, show_progress=false)

fig_diagnostics = Figure(size=(1350, 380))
ax_memory = Axis(fig_diagnostics[1, 1]; title="Memory: recall",
    xlabel="lag (samples)", ylabel="test R²", limits=(nothing, (0, 1.02)))
lines!(ax_memory, memory_diag.lags, memory_diag.r2; color=:darkcyan, linewidth=2)
band!(ax_memory, memory_diag.lags, zeros(length(memory_diag.r2)), memory_diag.r2;
    color=(:darkcyan, 0.15))
text!(ax_memory, 0.97, 0.95; text="recall sum = $(round(memory_diag.mc, digits=2))",
    space=:relative, align=(:right, :top))

δrelative = stability_diag.δ ./ max(first(stability_diag.δ), eps())
ax_stability = Axis(fig_diagnostics[1, 2]; title="Stability: perturbation response",
    xlabel="steps after perturbation", ylabel="log₁₀(distance / initial)")
lines!(ax_stability, stability_diag.times, log10.(δrelative .+ 1e-16);
    color=:black, linewidth=2)
hlines!(ax_stability, [0.0]; color=:grey60, linestyle=:dash)

ax_sep = Axis(fig_diagnostics[1, 3]; title="Separability: input-history distance",
    xlabel="history", ylabel="history",
    xticks=(1:3, ["A", "B", "C"]), yticks=(1:3, ["A", "B", "C"]))
sep_plot = heatmap!(ax_sep, separability_diag.pairwise_distances; colormap=:viridis)
Colorbar(fig_diagnostics[1, 4], sep_plot; label="standardised RMS distance")
Label(fig_diagnostics[0, 1:4],
    "$(uppercasefirst(String(selected_model))) reservoir diagnostics ($(replace(String(observation_choice), "_" => " ")))"; fontsize=22)
save("FIGURES/swarmRC/selected_$(selected_model)_$(observation_choice)_diagnostics.png", fig_diagnostics)
println("Memory recall sum: ", round(memory_diag.mc, digits=3))
println("Mean standardised separation: ", round(separability_diag.mean_pairwise_distance, digits=3))
println("Stability final/initial distance: ", round(last(δrelative), sigdigits=4))
fig_diagnostics

In [ ]:
if full_mode
if selected_model == :lymburn && observation_choice == :spatial_gaussian
    # The 3M Gaussian features are density, x-velocity-weighted density and
    # y-velocity-weighted density. Show density and the two velocity channels separately.
    M = size(observation_selected.C, 2)
    current_features = feature_map(res_selected, observation_selected)
    density_response = current_features[1:M]
    vx_response = current_features[M+1:2M]
    vy_response = current_features[2M+1:3M]
    velocity_response = hypot.(vx_response, vy_response)
    response_colormap = :viridis
    kernel_radii = kernel_sigma.(observation_selected.inv2w)
    rs = raw_state(res_selected)
    h_obs = swarm_pipeline.dynamics.lymburn.view_halfwidth

    fig_observation = Figure(size=(900, 430))
    ax_density = Axis(fig_observation[1, 1]; title="Gaussian density observations",
        xlabel="x", ylabel="y", aspect=DataAspect())
    scatter!(ax_density, rs.P[1, :], rs.P[2, :]; color=:grey75, markersize=3)
    density_plot = scatter!(ax_density, observation_selected.C[1, :], observation_selected.C[2, :];
        color=density_response, colormap=response_colormap, markersize=8)
    for m in 1:M
        centre = Point2f(observation_selected.C[1, m], observation_selected.C[2, m])
        lines!(ax_density, circle_points(centre, kernel_radii[m]); color=(:grey35, 0.12), linewidth=0.7)
    end
    xlims!(ax_density, -h_obs, h_obs); ylims!(ax_density, -h_obs, h_obs)
    Colorbar(fig_observation[2, 1], density_plot; vertical=false, label="density response")

    ax_velocity = Axis(fig_observation[1, 2]; title="Gaussian velocity observations",
        xlabel="x", ylabel="y", aspect=DataAspect())
    scatter!(ax_velocity, rs.P[1, :], rs.P[2, :]; color=:grey85, markersize=3)
    velocity_plot = scatter!(ax_velocity, observation_selected.C[1, :], observation_selected.C[2, :];
        color=velocity_response, colormap=response_colormap, markersize=8)
    for m in 1:M
        centre = Point2f(observation_selected.C[1, m], observation_selected.C[2, m])
        lines!(ax_velocity, circle_points(centre, kernel_radii[m]); color=(:grey35, 0.12), linewidth=0.7)
    end
    velocity_scale = (h_obs / 8) / max(maximum(velocity_response), eps())
    velocity_starts = [Point2f(observation_selected.C[1, m], observation_selected.C[2, m]) for m in 1:M]
    velocity_directions = [Vec2f(velocity_scale * vx_response[m],
        velocity_scale * vy_response[m]) for m in 1:M]
    arrows2d!(ax_velocity, velocity_starts, velocity_directions;
        color=(:darkcyan, 0.75), tiplength=5, shaftwidth=1.0)
    xlims!(ax_velocity, -h_obs, h_obs); ylims!(ax_velocity, -h_obs, h_obs)
    Colorbar(fig_observation[2, 2], velocity_plot; vertical=false, label="velocity-response magnitude")
elseif selected_model == :lymburn
    rs = raw_state(res_selected)
    fig_observation = Figure(size=(520, 480))
    ax_raw = Axis(fig_observation[1, 1]; title="Raw particle-position observation", xlabel="x", ylabel="y", aspect=DataAspect())
    raw_points = scatter!(ax_raw, rs.P[1, :], rs.P[2, :]; color=1:P_selected.N,
        colormap=:viridis, markersize=7)
    h_raw = swarm_pipeline.dynamics.lymburn.view_halfwidth
    xlims!(ax_raw, -h_raw, h_raw); ylims!(ax_raw, -h_raw, h_raw)
    Colorbar(fig_observation[1, 2], raw_points; label="agent index")
    fig_observation
elseif selected_model == :mizzi
    prey_now = res_selected.coupling isa NoCoupling ? nothing : out_driven_demo.pred_hist[end]
    fig_observation = plot_Mizzi_state(res_selected; prey=prey_now)
elseif selected_model == :lund
    fig_observation = Figure(size=(900, 360))
    ax_state = Axis(fig_observation[1, 1]; title="Lund reservoir state", xlabel="x", ylabel="y", aspect=DataAspect())
    lund_state = raw_state(res_selected)
    dispersed_now = res_selected.state.agent_state .== 0
    scatter!(ax_state, lund_state.P[1, dispersed_now], lund_state.P[2, dispersed_now]; color=:dodgerblue, markersize=7, label="state 0: dispersed")
    scatter!(ax_state, lund_state.P[1, .!dispersed_now], lund_state.P[2, .!dispersed_now]; color=:firebrick, markersize=7, label="state 1: clustered")
    xlims!(ax_state, 0, P_selected.L); ylims!(ax_state, 0, P_selected.L); axislegend(ax_state; position=:rt)
    ax_activity = Axis(fig_observation[1, 2]; title="Internal-state activity", xlabel="sample", ylabel="clustered fraction")
    clustered_fraction = res_selected.coupling isa NoCoupling ?
        [mean(s) for s in out_undriven.state_hist] :
        [mean(st.agent_state) for st in out_driven_demo.states]
    lines!(ax_activity, eachindex(clustered_fraction), clustered_fraction; color=:firebrick, linewidth=2)
    ylims!(ax_activity, -0.05, 1.05)
    fig_observation
else
    error("Unknown selected model: $selected_model")
end
save("FIGURES/swarmRC/selected_$(selected_model)_$(observation_choice)_observation.png", fig_observation)
fig_observation
else
    println("Skipped in results-only mode; set student_choice.run_mode=:full to generate this optional result.")
end


In [ ]:
if full_mode
if selected_model == :lymburn && observation_choice == :spatial_gaussian
    M = size(observation_selected.C, 2)
    kernel_importance = [
        norm(Wout_selected[:, 1 .+ [m, M + m, 2M + m]]) for m in 1:M
    ]
    kernel_importance ./= maximum(kernel_importance) + eps()

    rs = raw_state(res_selected)
    fig_importance = Figure(size=(700, 560))
    ax = Axis(fig_importance[1, 1], xlabel="x", ylabel="y",
        title="Which spatial observations does the readout use?", aspect=DataAspect())
    scatter!(ax, rs.P[1, :], rs.P[2, :]; color=:grey75, markersize=5)
    kp = scatter!(ax, observation_selected.C[1, :], observation_selected.C[2, :];
        color=kernel_importance, colormap=:magma,
        markersize=6 .+ 24 .* sqrt.(kernel_importance),
        strokecolor=:black, strokewidth=0.4)
    h_importance = swarm_pipeline.dynamics.lymburn.view_halfwidth
    xlims!(ax, -h_importance, h_importance); ylims!(ax, -h_importance, h_importance)
    Colorbar(fig_importance[1, 2], kp; label="readout importance")
    save("FIGURES/swarmRC/selected_lymburn_spatial_gaussian_importance.png", fig_importance)
    fig_importance
elseif selected_model == :lymburn
    raw_importance = vec(abs.(Wout_selected[:, 2:end]))
    (mean_position_importance=mean(raw_importance), maximum_position_importance=maximum(raw_importance))
elseif selected_model == :mizzi
    feature_importance = vec(abs.(Wout_selected[:, 2:end]))
    N = P_selected.N
    component_importance = (
        x_position=mean(feature_importance[1:N]),
        y_position=mean(feature_importance[N+1:2N]),
        x_velocity=mean(feature_importance[2N+1:3N]),
        y_velocity=mean(feature_importance[3N+1:4N]),
    )
    component_importance
elseif selected_model == :lund
    lund_feature_names = ["hull area", "pair distance", "alignment", "mean speed", "speed spread", "clustered fraction", "angular momentum", "speed (state 0)", "speed (state 1)"]
    feature_importance = vec(abs.(Wout_selected[:, 2:end]))
    feature_importance ./= max(maximum(feature_importance), eps())
    fig_importance = Figure(size=(900, 420))
    ax_importance = Axis(fig_importance[1, 1]; title="Lund observation importance", ylabel="normalised |readout weight|", xticks=(1:length(lund_feature_names), lund_feature_names), xticklabelrotation=pi/4)
    barplot!(ax_importance, 1:length(feature_importance), feature_importance; color=:darkcyan)
    fig_importance
else
    error("Unknown selected model: $selected_model")
end
else
    println("Skipped in results-only mode; set student_choice.run_mode=:full to generate this optional result.")
end


## ▶ Complete pipeline at a glance

This is the second talk shortcut (▶).

Run the notebook through training before running the next cell. It displays the input and coupling, swarm dynamics, observation layer and test prediction in one figure.

Each readout panel shows exactly 300 test samples. This spans 6 Lorenz time units for **Lymburn** and **Mizzi** ($\Delta t=0.02$), but 30 for **Lund** ($\Delta t=0.1$), so a different number of Lorenz oscillations is expected.

Typical clean-run times on my machine are:

| Tutorial preset | `:results_only` | `:full` |
|---|---:|---:|
| **Lymburn** | about 3 min | about 39 min |
| **Mizzi** | about 2 min | about 26 min |
| **Lund** | about 3 min | about 13 min |
| **Lund paper dynamics** | about 3 min | about 8 min |

Times include Julia start-up and compilation.


In [ ]:
pipeline_is_live = (@isdefined selected_model) && (@isdefined observation_choice) &&
    (@isdefined Yte_selected) && (@isdefined Yhat_selected) &&
    (@isdefined pipeline_snapshot_res) && selected_model == choice.swarm &&
    observation_choice == choice.observation

if pipeline_is_live
if selected_model == :lymburn && observation_choice == :spatial_gaussian
    teal = :darkcyan
    fig_pipeline = Figure(size=(1800, 480))
    colgap!(fig_pipeline.layout, 6)
    Label(fig_pipeline[0, 1:5], "Swarm-reservoir pipeline: one signal through four stages"; fontsize=24)

    # 1. Input and coupling: the two Lorenz coordinates form the predator path.
    state_input_index = pipeline_snapshot_index
    input_start = max(1, state_input_index - 999)
    ax_input = Axis(fig_pipeline[1, 1]; title="1  Input and coupling",
        xlabel="Lorenz x", ylabel="Lorenz y", aspect=DataAspect())
    lines!(ax_input, U[1, input_start:state_input_index], U[2, input_start:state_input_index]; color=teal, linewidth=2)
    scatter!(ax_input, [U[1, state_input_index]], [U[2, state_input_index]];
        color=:red, marker=:star5, markersize=18, label="predator")
    axislegend(ax_input; position=:rb)

    # 2. Reservoir: show the current swarm state and headings.
    state_now = raw_state(pipeline_snapshot_res)
    ax_swarm = Axis(fig_pipeline[1, 2]; title="2  Driven swarm\nsample=$(state_input_index), t=$(round(pipeline_snapshot_time, digits=2)), N=$(P_selected.N)",
        xlabel="x", ylabel="y", aspect=DataAspect())
    swarm_heading = atan.(state_now.V[2, :], state_now.V[1, :])
    swarm_plot = scatter!(ax_swarm, state_now.P[1, :], state_now.P[2, :];
        color=swarm_heading, colormap=heading_colormap, colorrange=(-pi, pi), markersize=5)
    h = swarm_pipeline.dynamics.lymburn.view_halfwidth
    arrow_length = h / 15
    arrow_positions = [Point2f(state_now.P[1, i], state_now.P[2, i]) for i in 1:P_selected.N]
    arrow_directions = [begin
        speed = hypot(state_now.V[1, i], state_now.V[2, i])
        Vec2f(arrow_length * state_now.V[1, i] / max(speed, eps()),
              arrow_length * state_now.V[2, i] / max(speed, eps()))
    end for i in 1:P_selected.N]
    arrows2d!(ax_swarm, arrow_positions, arrow_directions;
        color=swarm_heading, colormap=heading_colormap, colorrange=(-pi, pi),
        tiplength=5, shaftwidth=1.0)
    xlims!(ax_swarm, -h, h); ylims!(ax_swarm, -h, h)
    Colorbar(fig_pipeline[2, 2], swarm_plot; vertical=false,
        ticks=([-pi, 0, pi], ["−π", "0", "π"]), label="heading")

    # 3. Observation: M density features and 2M velocity-weighted features.
    M = size(observation_selected.C, 2)
    current_observation = feature_map(pipeline_snapshot_res, observation_selected)
    kernel_density = current_observation[1:M]
    kernel_vx = current_observation[M+1:2M]
    kernel_vy = current_observation[2M+1:3M]
    kernel_velocity = hypot.(kernel_vx, kernel_vy)
    density_scaled = kernel_density ./ max(maximum(abs.(kernel_density)), eps())
    response_colormap = :viridis
    ax_obs = Axis(fig_pipeline[1, 3]; title="3a  Gaussian density",
        xlabel="x", ylabel="y", aspect=DataAspect())
    scatter!(ax_obs, state_now.P[1, :], state_now.P[2, :]; color=:grey80, markersize=3)
    kernel_plot = scatter!(ax_obs, observation_selected.C[1, :], observation_selected.C[2, :];
        color=density_scaled, colormap=response_colormap, colorrange=(0, 1), markersize=7)
    xlims!(ax_obs, -h, h); ylims!(ax_obs, -h, h)
    Colorbar(fig_pipeline[2, 3], kernel_plot; vertical=false, label="density response")

    ax_obs_velocity = Axis(fig_pipeline[1, 4]; title="3b  Gaussian velocity",
        xlabel="x", ylabel="y", aspect=DataAspect())
    scatter!(ax_obs_velocity, state_now.P[1, :], state_now.P[2, :]; color=:grey85, markersize=3)
    velocity_plot = scatter!(ax_obs_velocity, observation_selected.C[1, :], observation_selected.C[2, :];
        color=kernel_velocity, colormap=response_colormap, markersize=7)
    velocity_scale = (h / 8) / max(maximum(kernel_velocity), eps())
    kernel_starts = [Point2f(observation_selected.C[1, m], observation_selected.C[2, m]) for m in 1:M]
    kernel_directions = [Vec2f(velocity_scale * kernel_vx[m],
        velocity_scale * kernel_vy[m]) for m in 1:M]
    arrows2d!(ax_obs_velocity, kernel_starts, kernel_directions;
        color=(:darkcyan, 0.75), tiplength=5, shaftwidth=1.0)
    xlims!(ax_obs_velocity, -h, h); ylims!(ax_obs_velocity, -h, h)
    Colorbar(fig_pipeline[2, 4], velocity_plot; vertical=false, label="velocity-response magnitude")

    # 4. Readout: compare observations on a short section of unseen test data.
    @assert length(Yte_selected) >= 300 "The pipeline summary requires 300 test samples."
    nshow = 300
    ax_readout = Axis(fig_pipeline[1, 5]; title="4  Linear readout",
        xlabel="test sample (Δt=$(dt))", ylabel="target")
    lines!(ax_readout, 1:nshow, vec(Yte_selected)[1:nshow];
        color=teal, linewidth=2, label="actual Lorenz x")
    lines!(ax_readout, 1:nshow, vec(Yhat_selected)[1:nshow];
        color=:black, linestyle=:dash, label="prediction")
    axislegend(ax_readout; position=:lb)
    Label(fig_pipeline[2, 5], "Gaussian R=$(round(R_selected, digits=3))   raw positions R=$(round(R_raw, digits=3))";
        fontsize=15)

    for col in 1:5; colsize!(fig_pipeline.layout, col, Relative(1 / 5)); end
    save("FIGURES/swarmRC/lymburn_pipeline_summary.png", fig_pipeline)
    fig_pipeline
elseif selected_model == :mizzi && observation_choice == :raw_state
    teal = :darkcyan
    fig_pipeline = Figure(size=(1800, 480))
    colgap!(fig_pipeline.layout, 6)
    Label(fig_pipeline[0, 1:5], "Mizzi tutorial pipeline: prey to territorial reservoir to readout"; fontsize=24)

    state_input_index = pipeline_snapshot_index
    input_start = max(1, state_input_index - 999)
    ax_input = Axis(fig_pipeline[1, 1]; title="1  Input and coupling",
        xlabel="u(t−τ)", ylabel="u(t)", aspect=DataAspect())
    lines!(ax_input, U[1, input_start:state_input_index], U[2, input_start:state_input_index]; color=teal, linewidth=2)
    scatter!(ax_input, [U[1, state_input_index]], [U[2, state_input_index]];
        color=:dodgerblue, markersize=12, label="prey")
    axislegend(ax_input; position=:rb)

    state_now = raw_state(pipeline_snapshot_res)
    ax_swarm = Axis(fig_pipeline[1, 2]; title="2  Territorial reservoir\nsample=$(state_input_index), t=$(round(pipeline_snapshot_time, digits=2)), N=$(P_selected.N)",
        xlabel="x", ylabel="y", aspect=DataAspect())
    plot_Mizzi_territories!(ax_swarm, P_selected)
    scatter!(ax_swarm, state_now.P[1, :], state_now.P[2, :];
        color=:firebrick, markersize=7, label="agents")
    scatter!(ax_swarm, [U[1, state_input_index]], [U[2, state_input_index]];
        color=:dodgerblue, markersize=11, label="prey")
    h = P_selected.plot_extent
    xlims!(ax_swarm, -h, h); ylims!(ax_swarm, -h, h)

    raw_now = feature_map(pipeline_snapshot_res)
    # Makie expects z[agent, component] for the supplied x and y coordinates.
    state_components = reshape(raw_now, P_selected.N, 4)
    ax_obs = Axis(fig_pipeline[1, 3:4]; title="3  Home-relative observation",
        xlabel="agent", ylabel="state component",
        yticks=(1:4, ["Δx", "Δy", "vx", "vy"]))
    observation_limit = max(maximum(abs, state_components), eps())
    obs_plot = heatmap!(ax_obs, 1:P_selected.N, 1:4, state_components;
        colormap=:vik, colorrange=(-observation_limit, observation_limit))
    Colorbar(fig_pipeline[2, 3:4], obs_plot; vertical=false, label="feature value")

    @assert length(Yte_selected) >= 300 "The pipeline summary requires 300 test samples."
    nshow = 300
    ax_readout = Axis(fig_pipeline[1, 5]; title="4  Linear readout",
        xlabel="test sample (Δt=$(dt))", ylabel="target")
    lines!(ax_readout, 1:nshow, vec(Yte_selected)[1:nshow];
        color=teal, linewidth=2, label="actual")
    lines!(ax_readout, 1:nshow, vec(Yhat_selected)[1:nshow];
        color=:black, linestyle=:dash, label="prediction")
    axislegend(ax_readout; position=:rb, labelsize=12)
    Label(fig_pipeline[2, 5], "direct-state R=$(round(R_selected, digits=3))"; fontsize=15)
    for col in 1:5; colsize!(fig_pipeline.layout, col, Relative(1 / 5)); end
    save("FIGURES/swarmRC/mizzi_pipeline_summary.png", fig_pipeline)
    fig_pipeline
elseif selected_model == :lund && observation_choice == :lund_aggregate
    teal = :darkcyan
    fig_pipeline = Figure(size=(1800, 480))
    colgap!(fig_pipeline.layout, 6)
    Label(fig_pipeline[0, 1:5], "Lund tutorial pipeline: temperature to two-state swarm to readout"; fontsize=24)

    state_input_index = pipeline_snapshot_index
    input_start = max(1, state_input_index - 299)
    ax_input = Axis(fig_pipeline[1, 1]; title="1  Input and coupling",
        xlabel="sample", ylabel="standardised Lorenz x")
    lines!(ax_input, input_start:state_input_index, vec(U[1, input_start:state_input_index]); color=teal, linewidth=2)

    state_now = raw_state(pipeline_snapshot_res)
    ax_swarm = Axis(fig_pipeline[1, 2]; title="2  Two-state reservoir\nsample=$(state_input_index), t=$(round(pipeline_snapshot_time, digits=2)), N=$(P_selected.N)",
        xlabel="x", ylabel="y", aspect=DataAspect())
    dispersed = state_now.agent_state .== 0
    scatter!(ax_swarm, state_now.P[1, dispersed], state_now.P[2, dispersed];
        color=:dodgerblue, markersize=6, label="dispersed")
    scatter!(ax_swarm, state_now.P[1, .!dispersed], state_now.P[2, .!dispersed];
        color=:firebrick, markersize=6, label="clustered")
    xlims!(ax_swarm, 0, P_selected.L); ylims!(ax_swarm, 0, P_selected.L)
    axislegend(ax_swarm; position=:rt)

    aggregate_now = feature_map(pipeline_snapshot_res)
    aggregate_names = ["hull", "distance", "alignment", "speed",
        "speed spread", "clustered", "rotation", "speed 0", "speed 1"]
    train_range = shift + washout + 1:shift + train_len
    aggregate_mean = vec(mean(Xselected[:, train_range]; dims=2))
    aggregate_std = vec(std(Xselected[:, train_range]; dims=2)) .+ eps()
    aggregate_scaled = (aggregate_now .- aggregate_mean) ./ aggregate_std
    ax_obs = Axis(fig_pipeline[1, 3:4]; title="3  Aggregate observation",
        ylabel="standardised current value",
        xticks=(1:9, aggregate_names), xticklabelrotation=pi/3)
    barplot!(ax_obs, 1:9, aggregate_scaled; color=teal)

    @assert length(Yte_selected) >= 300 "The pipeline summary requires 300 test samples."
    nshow = 300
    ax_readout = Axis(fig_pipeline[1, 5]; title="4  Linear readout",
        xlabel="test sample (Δt=$(dt))", ylabel="target")
    lines!(ax_readout, 1:nshow, vec(Yte_selected)[1:nshow];
        color=teal, linewidth=2, label="actual Lorenz x")
    lines!(ax_readout, 1:nshow, vec(Yhat_selected)[1:nshow];
        color=:black, linestyle=:dash, label="prediction")
    axislegend(ax_readout; position=:lb)
    Label(fig_pipeline[2, 5], "aggregate-state R=$(round(R_selected, digits=3))"; fontsize=15)
    for col in 1:5; colsize!(fig_pipeline.layout, col, Relative(1 / 5)); end
    save("FIGURES/swarmRC/lund_pipeline_summary.png", fig_pipeline)
    fig_pipeline
else
    println("A pipeline summary is implemented for each tutorial preset's default observation.")
end
else
    error("Run the notebook through the training section before displaying the pipeline summary.")
end


## Next steps

Play!

Change one component at a time:

1. select a different swarm regime or model;
2. change the input signal or coupling;
3. replace or augment the spatial observation;
4. run `validate_pipeline` and compare performance on unseen test data against this baseline.

[`Tutorial_TDA.ipynb`](./Tutorial_TDA.ipynb) shows how topological summaries can be constructed from swarm states. Connecting those summaries to this training pipeline is a future extension; it is not a selectable option in this tutorial.

Use `advanced/Existing_Model_Validation.ipynb` for published numerical checks. [`README.md`](./README.md) describes the code layout; [`MODEL_IMPLEMENTATION_COMPARISON.md`](./MODEL_IMPLEMENTATION_COMPARISON.md) records where the tutorial differs from the source papers.